In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from scipy.spatial import KDTree 

In [ ]:
def get_binary_image(img):
    """
    Converts RGB image to Binary using tuned Adaptive Thresholding.
    Matches 'processor.py' logic.
    """
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img
        
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # Tuned parameters: Block 55, Constant 7
    adaptive_result = cv2.adaptiveThreshold(
        blurred, 255, 
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
        cv2.THRESH_BINARY_INV, 55, 7
    )
    return adaptive_result

def get_pin_coordinates(binary_img):
    """
    Finds pin centers using Hough Circle Transform + Robust KDTree Filtering.
    Matches 'processor.py' logic.
    """
    # 1. HOUGH TRANSFORM
    circles = cv2.HoughCircles(
        binary_img, 
        cv2.HOUGH_GRADIENT, 
        dp=1,           
        minDist=12,     # Pins are rarely closer than 12px
        param1=50,      
        param2=10,      # HIGHER = Less sensitive, fewer false positives
        minRadius=4,    
        maxRadius=15    
    )
    
    if circles is None:
        return []

    # Get raw coordinates
    raw_coords = np.round(circles[0, :]).astype("int")[:, :2]
    
    # Safety check for low detection counts
    if len(raw_coords) < 15:
        return raw_coords.tolist()

    # 2. CALCULATE GRID PITCH & FILTER
    tree = KDTree(raw_coords)
    dist, _ = tree.query(raw_coords, k=2) 
    neighbor_distances = dist[:, 1]
    
    # Estimate pitch (median distance)
    estimated_pitch = np.median(neighbor_distances)
    
    filtered_coords = []
    
    # Tolerances based on estimated pitch
    min_dist_tol = estimated_pitch * 0.6
    max_dist_tol = estimated_pitch * 1.5
    search_radius = estimated_pitch * 2.5
    
    for i, (x, y) in enumerate(raw_coords):
        d = neighbor_distances[i]
        
        # Check 1: Distance to nearest neighbor
        if d < min_dist_tol or d > max_dist_tol:
            continue
            
        # Check 2: Local Density (Keep corners with >= 2 neighbors)
        neighbors_idx = tree.query_ball_point([x, y], r=search_radius)
        num_neighbors = len(neighbors_idx) - 1 
        
        if num_neighbors >= 2:
            filtered_coords.append((x, y))
            
    # 3. SAFETY NET
    # If filter removes >50% of points, assume it failed and return raw
    if len(filtered_coords) < (len(raw_coords) * 0.5):
        print("Warning: Filtering removed too many points. Reverting to raw.")
        return raw_coords.tolist()
            
    return filtered_coords if filtered_coords else raw_coords.tolist()

def extract_pins(img, coords, crop_size=32, target_size=256):
    """
    Crops pins, applies CLAHE enhancement, and resizes.
    Matches 'processor.py' logic.
    """
    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = img

    # Enhancement Step (Important for consistency with App)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(gray)
    enhanced = cv2.GaussianBlur(enhanced, (5, 5), 0)
    
    pins = []
    h, w = enhanced.shape
    
    for i, (cx, cy) in enumerate(coords):
        half = crop_size // 2
        # Boundary checks
        if (cy - half >= 0 and cx - half >= 0 and 
            cy + half < h and cx + half < w):
            
            crop = enhanced[cy-half:cy+half, cx-half:cx+half]
            
            # Resize to model input size (256x256)
            crop_resized = cv2.resize(crop, (target_size, target_size), interpolation=cv2.INTER_CUBIC)
            
            # Convert to RGB (Anomalib expects 3 channels)
            crop_rgb = cv2.cvtColor(crop_resized, cv2.COLOR_GRAY2RGB)
            
            pins.append({
                "id": i,
                "crop": crop_rgb,
                "coords": (cx, cy)
            })
            
    return pins

In [ ]:
# Directory containing your raw test images
image_dir = "./data/7_1_26/" 

# Create output directory for checking
os.makedirs("output_debug", exist_ok=True)

for filename in os.listdir(image_dir):
    if not (filename.endswith(".jpg") or filename.endswith(".png")):
        continue
        
    img_path = os.path.join(image_dir, filename)
    img = cv2.imread(img_path)
    
    # 1. Get Binary
    binary = get_binary_image(img)
    
    # 2. Get Coords
    coords = get_pin_coordinates(binary)
    
    # 3. Extract Pins
    pins_data = extract_pins(img, coords, crop_size=40) # Ensure crop_size matches main.py

    # Save pins to debug folder
    """ for i, pin_data in enumerate(pins_data):
        cv2.imwrite(f"output_debug/{filename}_{i}.png", pin_data['crop']) """
    
    print(f"Image: {filename} | Pins Detected: {len(pins_data)}")
    
    # --- VISUALIZATION (Optional) ---
    # Draw circles on original image to verify detection
    vis_img = img.copy()
    for (x, y) in coords:
        cv2.circle(vis_img, (x, y), 5, (0, 255, 0), 2)
        
    # Save visualization
    cv2.imwrite(f"output_debug/checked_{filename}", vis_img)
    
    # Show one pin example
    if len(pins_data) > 0:
        plt.figure(figsize=(3,3))
        plt.imshow(pins_data[0]['crop'])
        plt.title(f"First Pin of {filename}")
        plt.axis('off')
        plt.show()

In [ ]:
def check_pin_visibility(image_path):
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Gaussian Blur (Common step)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # Adaptive Threshold
    # Block Size 25, Constant 10 (Fine tune these!)
    adaptive_result = cv2.adaptiveThreshold(blurred, 255, 
                                            cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                            cv2.THRESH_BINARY_INV, 55, 7)

    if os.path.exists("output"):
        cv2.imwrite(f"output/adaptive_{image_path.split('/')[-1]}", adaptive_result)
    else:
        os.mkdir("output")
        cv2.imwrite(f"output/adaptive_{image_path.split('/')[-1]}", adaptive_result)

for img in os.listdir("./data/socket/"):
    check_pin_visibility(f"./data/socket/{img}")

In [ ]:
def generate_theoretical_grid(start_point, spacing, rows, cols):
    x_start, y_start = start_point
    dx, dy = spacing
    
    theoretical_coords = []
    for r in range(rows):
        for c in range(cols):
            x = x_start + (c * dx)
            y = y_start + (r * dy)
            theoretical_coords.append((int(x), int(y)))
            
    return theoretical_coords


def extract_all_potential_candidates(image, grid_spacing, margin=5, theoretical_grid=None):
    candidates = []
    # theoretical_grid is derived from your Hough Line intersections
    for (x, y) in theoretical_grid:
        patch = image[y-margin:y+margin, x-margin:x+margin]
        candidates.append({
            "coords": (x, y),
            "patch": patch
        })
    return candidates 


In [ ]:
""" def extract_aligned_pins(image_path, coords, crop_size=64):  
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Gentler CLAHE for better details
    clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(4,4))  # Reduced from (8,8)
    enhanced = clahe.apply(gray)
    
    # Add bilateral filtering to preserve edges
    enhanced = cv2.bilateralFilter(enhanced, 5, 50, 50)
    
    pins = []
    for i, (cx, cy) in enumerate(coords):
        half = crop_size // 2
        if (cy-half >= 0 and cx-half >= 0 and 
            cy+half < enhanced.shape[0] and cx+half < enhanced.shape[1]):
            crop = enhanced[cy-half:cy+half, cx-half:cx+half]
            # Resize to consistent size
            crop = cv2.resize(crop, (256, 256), interpolation=cv2.INTER_CUBIC)
            pins.append((crop, i, cx, cy))
    return pins """

def analyze_data_quality(pins):
    quality_issues = []
    for crop, idx, x, y in pins:
        # Check variance (blurriness)
        variance = cv2.Laplacian(crop, cv2.CV_64F).var()
        if variance < 50:
            quality_issues.append(f"Pin {idx} at ({x},{y}) may be blurry (var={variance:.2f})")
        
        # Check brightness
        mean_brightness = crop.mean()
        if mean_brightness < 30 or mean_brightness > 225:
            quality_issues.append(f"Pin {idx} at ({x},{y}) has extreme brightness ({mean_brightness:.2f})")
    return quality_issues

In [ ]:
def extract_precise_grid_hough(binary_path):
    binary = cv2.imread(binary_path, cv2.IMREAD_GRAYSCALE)
    
    circles = cv2.HoughCircles(
        binary, 
        cv2.HOUGH_GRADIENT, 
        dp=1,           
        minDist=12,     # Pins are rarely closer than 12px
        param1=60,      
        param2=10,      # HIGHER = Less sensitive, fewer false positives
        minRadius=4,    
        maxRadius=15    
    )
    
    coords = []
    if circles is not None:
        circles = np.round(circles[0, :]).astype("int")
        coords = [(x, y) for x, y in circles[:, :2]]
        print(f"Found {len(coords)} circles with Hough")
    else:
        print("No circles found with Hough!")
    
    return coords

def extract_precise_grid(binary_path):
    binary = cv2.imread(binary_path, cv2.IMREAD_GRAYSCALE)
    
    template_size = 30  # Bigger template
    template = np.zeros((template_size, template_size), dtype=np.uint8)
    cv2.circle(template, (template_size//2, template_size//2), 8, 255, -1)  # Solid circle
    cv2.circle(template, (template_size//2, template_size//2), 12, 0, 3)    # Black ring around it
    
    result = cv2.matchTemplate(binary, template, cv2.TM_CCOEFF_NORMED)
    
    # MUCH higher threshold
    threshold = 0.8  # Increased from 0.7
    locations = np.where(result >= threshold)
    
    # Remove duplicate detections (Non-Maximum Suppression)
    coords_raw = list(zip(locations[1], locations[0]))
    
    # Filter out coordinates that are too close to each other
    coords = []
    min_distance = 20  # Minimum pixels between pin centers
    
    for x, y in coords_raw:
        # Check if this coord is far enough from existing ones
        too_close = False
        for existing_x, existing_y in coords:
            distance = np.sqrt((x - existing_x)**2 + (y - existing_y)**2)
            if distance < min_distance:
                too_close = True
                break
        
        if not too_close:
            coords.append((x, y))
    
    print(f"Found {len(coords)} pins after filtering")
    return coords

def filter_pins_simple_distance(coords, max_neighbor_dist=200):
    filtered_coords = []
    coords_array = np.array(coords)
    
    for i, (x, y) in enumerate(coords):
        # Count neighbors within max_neighbor_dist
        distances = np.sqrt(np.sum((coords_array - [x, y])**2, axis=1))
        neighbors = np.sum(distances < max_neighbor_dist) - 1  # -1 to exclude self
        
        # Keep pins that have at least 8 neighbors (socket pins are densely packed)
        if neighbors >= 8:
            filtered_coords.append((x, y))
    
    print(f"Distance filtering: {len(coords)} -> {len(filtered_coords)} pins")
    return filtered_coords

def extract_aligned_pins(image_path, coords, crop_size=32):
    # Extract pins with better preprocessing
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # CLAHE for better contrast
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(gray)
    
    pins = []
    for i, (cx, cy) in enumerate(coords):
        half = crop_size // 2
        if (cy-half >= 0 and cx-half >= 0 and cy+half < enhanced.shape[0] and cx+half < enhanced.shape[1]): # Boundary check (too close to the edge)
            crop = enhanced[cy-half:cy+half, cx-half:cx+half]
            # Normalize each pin individually
            crop = cv2.normalize(crop, None, 0, 255, cv2.NORM_MINMAX)
            crop = cv2.GaussianBlur(crop, (3,3), 0)
            pins.append((crop, i, cx, cy))

    return pins 

In [ ]:
def filter_good_pins(pins, variance_threshold=100):
    """Remove blurry/bad quality pins and not pins (outside pins area in socket)"""
    good_pins = []
    
    """ for crop, idx, x, y in pins:
        # Check if pin is in focus (Laplacian variance)
        variance = cv2.Laplacian(crop, cv2.CV_64F).var()
        
        if variance > variance_threshold:
            good_pins.append((crop, idx, x, y)) """
    
    good_pins = pins
    
    max_dist_pins = 20 # max distance from one pins center to another
    final_good_pins = []
    for i in range(len(good_pins)):
        crop_i, idx_i, x_i, y_i = good_pins[i]
        too_close = False
        for j in range(len(final_good_pins)):
            crop_j, idx_j, x_j, y_j = final_good_pins[j]
            dist = np.sqrt((x_i - x_j)**2 + (y_i - y_j)**2)
            if dist < max_dist_pins: 
                too_close = True 
                break
        if not too_close:
            final_good_pins.append(good_pins[i])
    good_pins = final_good_pins
    print(f"Filtered {len(pins)} -> {len(good_pins)} quality pins")
    return good_pins

def visualize_pin_quality(pins):
    fig, axes = plt.subplots(4, 8, figsize=(16, 8))
    for i, (crop, idx, x, y) in enumerate(pins[:32]):
        row, col = i // 8, i % 8
        axes[row, col].imshow(crop, cmap='gray')
        axes[row, col].set_title(f"{idx}")
        axes[row, col].axis('off')
    plt.show()

def visualize_detection_on_original(image_path, pins):
    img = cv2.imread(image_path)
    vis_img = img.copy()
    crop_size = 32
    coords = [(x, y) for (_, _, x, y) in pins]

    # Draw Bounding Boxes
    half_size = crop_size // 2
    h, w = vis_img.shape[:2] # Get image dimensions

    for (x, y) in coords:
        # Calculate bounding box coordinates
        x1 = x - half_size
        y1 = y - half_size
        x2 = x + half_size
        y2 = y + half_size

        # Green (0, 255, 0), Thickness 2 px
        cv2.rectangle(vis_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
        # Optional: Draw a small red dot at the exact center
        cv2.circle(vis_img, (x, y), 2, (0, 0, 255), -1)
        
    # Convert BGR (OpenCV) to RGB (Matplotlib)
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB))
    plt.title(f"Detection Evaluation: {len(coords)} Pins Detected")
    plt.axis('off')
    plt.show()
    
    return vis_img

In [ ]:
# Use the coordinates we found in the previous step
# coords = [(x1,y1), (x2,y2), ...] 

def create_anomalib_dataset(image_path, pins, img_name, output_dir="datasets/socket_pins"):
    img = cv2.imread(image_path)
    
    # Create folders
    train_dir = os.path.join(output_dir, "train/good")
    test_dir = os.path.join(output_dir, "test/good")
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)
    
    for i, (crop, idx, x, y) in enumerate(pins):
        if i < 0.8*len(pins):
            cv2.imwrite(os.path.join(train_dir, f"{img_name}_{idx}.png"), crop)
        else:
            cv2.imwrite(os.path.join(test_dir, f"{img_name}_{idx}.png"), crop)
            
    print(f"Dataset generated at {output_dir}")


for img in os.listdir("./data/socket/"):
    print(f"Processing {img}...")
    
    # Get coordinates from binary image
    binary_path = f"output/adaptive_{img}"
    coords = extract_precise_grid_hough(binary_path)
    # coords = filter_pins_simple_distance(coords, max_neighbor_dist=200)

    # Extract pins from original image
    original_path = f"./data/socket/{img}"
    pins = extract_aligned_pins(original_path, coords)
    
    # Filter and visualize
    pins = filter_good_pins(pins) # Need to work on this
    """ visualize_pin_quality(pins) """
    visualize_detection_on_original(original_path, pins)
    """ analyze_data_quality(pins) """
    
    # Create dataset
    create_anomalib_dataset(original_path, pins, img.split(".")[0])